In [ ]:
import jax
import jax.numpy as jnp
import mhmc
import matplotlib.pyplot as plt

def count_cycles(indices):
  num_cycles=0
  last_pos = 0
  def f(carry, x):
    (num_cycles, last_pos) = carry
    sensor_pos = jax.lax.select(last_pos == 0, len(x)-1, 0)
    tripped = x[sensor_pos] == 0
    num_cycles = num_cycles + jnp.logical_and(tripped, sensor_pos == 0)
    last_pos = jax.lax.select(tripped, sensor_pos, last_pos)
    return (num_cycles, last_pos), num_cycles
  (cycles,_), num_cycles_plt = jax.lax.scan(f, (num_cycles, last_pos), indices)
  return cycles, num_cycles_plt


def sigmoid(x, u, s):
  return jnp.exp(-0.5*(x-u)*(x-u)/(s*s))/(jnp.sqrt(2*jnp.pi)*s)

U = 3
S = 0.5

def f(x):
  return (0.5*sigmoid(x, -U, S)+0.5*sigmoid(x, U, S))

def distribution_fn(x, temp=1.0):
  beta = 1/temp
  x_int = jnp.linspace(-3*U, 3*U, 1000)
  normalizer = jnp.trapezoid(f(x_int)**beta, x_int)
  return f(x)**beta/normalizer

def log_dfun(x):
  return jnp.log(sigmoid(x, -U, S)+sigmoid(x, U, S))

def perterb_fn(key, x):
  delta = jax.random.normal(key)*0.5
  return x+delta


NUM_SAMPLES=10000

temperatures=jnp.linspace(0.1, 4.0, 3)
visit_state = {
  'rejections': jnp.zeros(len(temperatures)-1),
  'samples' : jnp.zeros((NUM_SAMPLES, len(temperatures))),
  'count': 0,
}

step_fn, step_state = mhmc.make_metropolis_hastings_step(log_dfun, perterb_fn, jnp.zeros(1)+2.0)
step_states = mhmc.replicate_tree(temperatures.size, step_state)

def cs(step_state, visit_state, parallel_tempering_info, step):
  for i in range(temperatures.size):
    visit_state['samples'] = visit_state['samples'].at[step,i].set(step_state['x'][i][0])
  visit_state['rejections'] = visit_state['rejections'] + (1-jnp.exp(parallel_tempering_info['log_alphas']))
  visit_state['count'] += 1
  return visit_state

key = jax.random.key(0)
scan_body, state = mhmc.parallel_tempering_scan_fn(key, step_fn, step_states, temperatures, visit_fn=cs, visit_state=visit_state)

state['temperatures'] = temperatures
state['visit_state']['rejections'] = jnp.zeros_like(state['visit_state']['rejections'])
state['visit_state']['count'] = 0
state, indices = jax.lax.scan(scan_body, state, jnp.arange(NUM_SAMPLES))

visit_state = state['visit_state']

# c, _ = count_cycles(indices)
# fig=plt.figure()
# total, num_cycles_plt = count_cycles(indices)
# plt.plot(num_cycles_plt)
x=jnp.linspace(-10,10,1000)
for i,t in enumerate(temperatures):
  fig=plt.figure()
  ax=fig.gca()
  ax.plot(x, distribution_fn(x, temp=t))
  _=ax.hist(visit_state['samples'][:,i], density=True, bins=200)


In [ ]:
NUM_CYCLES = 10
NUM_SAMPLES = 1000

temperatures=jnp.linspace(0.01, 2.0, 4)
visit_state = {
  'rejections': jnp.zeros(len(temperatures)-1),
  'samples' : jnp.zeros((NUM_SAMPLES, len(temperatures))),
  'count': 0,
}

step_fn, step_state = mhmc.make_metropolis_hastings_step(log_dfun, perterb_fn, jnp.zeros(1)+2.0)
step_states = mhmc.replicate_tree(temperatures.size, step_state)


def optimize_schedule(temperatures, visit_state):
    rejection_rates = visit_state['rejections']/visit_state['count'] + 1e-8
    cum_barrier = jnp.concatenate([jnp.array([0.0]), jnp.cumsum(rejection_rates)])
    cum_barrier_norm = cum_barrier / cum_barrier[-1]
    N = rejection_rates.shape[0]
    target_barrier = jnp.linspace(0.0, 1.0, N + 1)
    return jnp.interp(target_barrier, cum_barrier_norm, temperatures)

key = jax.random.key(0)

scan_body, state = mhmc.parallel_tempering_scan_fn(key, step_fn, step_states, temperatures, visit_fn=cs, visit_state=visit_state)

temps = jnp.zeros((NUM_CYCLES, temperatures.size))
rejection_rates = jnp.zeros((NUM_CYCLES, temperatures.size-1))
cycle_count = jnp.zeros(NUM_CYCLES)
for cycle in range(NUM_CYCLES):
  state['temperatures'] = temperatures
  state['visit_state']['rejections'] = jnp.zeros_like(state['visit_state']['rejections'])
  state['visit_state']['count'] = 0
  state, indices = jax.lax.scan(scan_body, state, jnp.arange(NUM_SAMPLES))
  temperatures = optimize_schedule(temperatures, state['visit_state'])
  rejection_rates = rejection_rates.at[cycle, :].set(state['visit_state']['rejections']/state['visit_state']['count'])
  temps = temps.at[cycle, :].set(temperatures)
  c, _ = count_cycles(indices)
  cycle_count = cycle_count.at[cycle].set(c/NUM_SAMPLES)

plt.figure()
plt.plot(temps)
plt.figure()
plt.plot(rejection_rates)
plt.figure()
plt.plot(cycle_count)


In [ ]:
cycle_count.shape

In [ ]:


alphas
